In [1]:
import json
import os
from pathlib import Path
from pprint import pprint

import torch
import torch.nn as nn
import torch.optim as optim

from datasets import load_from_disk
from transformers import AutoModel, AutoTokenizer

from tqdm import tqdm
import wandb
import mlflow

In [2]:
model_path = "distilbert/distilbert-base-uncased"
model = AutoModel.from_pretrained(model_path)
tokenizer = AutoTokenizer.from_pretrained(model_path)

Loading weights:   0%|          | 0/100 [00:00<?, ?it/s]

[transformers] DistilBertModel LOAD REPORT from: distilbert/distilbert-base-uncased
Key                     | Status     |  | 
------------------------+------------+--+-
vocab_transform.bias    | UNEXPECTED |  | 
vocab_projector.bias    | UNEXPECTED |  | 
vocab_transform.weight  | UNEXPECTED |  | 
vocab_layer_norm.bias   | UNEXPECTED |  | 
vocab_layer_norm.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [3]:
data_dir = Path(os.getcwd()).parent/"data"
dataset = load_from_disk(data_dir/"updated_ai4privacy_300k_pii")
train_ds, val_ds = dataset["train"], dataset["validation"]

with open(data_dir/"label_info.json") as f:
    label_info: dict = json.load(f)

In [4]:
print(f"dataset column names: {train_ds.column_names}\n\n")
example = train_ds[0]
print("dataset example:")
pprint(example)

dataset column names: ['source_text', 'privacy_mask']


dataset example:
{'privacy_mask': [{'end': 297,
                   'label': 'USERNAME',
                   'start': 287,
                   'value': 'wynqvrh053'},
                  {'end': 318,
                   'label': 'TIME',
                   'start': 311,
                   'value': '10:20am'},
                  {'end': 330,
                   'label': 'USERNAME',
                   'start': 321,
                   'value': 'luka.burg'},
                  {'end': 346, 'label': 'TIME', 'start': 344, 'value': '21'},
                  {'end': 363,
                   'label': 'USERNAME',
                   'start': 349,
                   'value': 'qahil.wittauer'},
                  {'end': 392,
                   'label': 'TIME',
                   'start': 377,
                   'value': 'quarter past 13'},
                  {'end': 416,
                   'label': 'USERNAME',
                   'start': 395,
             

In [29]:
pprint(list(label_info.keys()))

['bio_tags',
 'entity_classes',
 'label2id',
 'id2label',
 'train_bio_counts',
 'description']


In [5]:
ex_tokenized = tokenizer(example["source_text"])
print(ex_tokenized)

{'input_ids': [101, 3395, 1024, 2177, 24732, 2005, 20247, 2832, 2204, 2851, 1010, 3071, 1010, 1045, 3246, 2023, 4471, 4858, 2017, 2092, 1012, 2004, 2057, 3613, 2256, 20247, 6194, 1010, 1045, 2052, 2066, 2000, 10651, 2017, 2006, 1996, 6745, 8973, 1998, 3145, 2592, 1012, 3531, 2424, 2917, 1996, 17060, 2005, 2256, 9046, 6295, 1024, 1011, 1059, 6038, 4160, 19716, 2232, 2692, 22275, 1011, 3116, 2012, 2184, 1024, 2322, 3286, 1011, 11320, 2912, 1012, 20934, 10623, 1011, 3116, 2012, 2538, 1011, 1053, 4430, 4014, 1012, 15966, 2696, 13094, 1011, 3116, 2012, 4284, 2627, 2410, 1011, 1043, 14854, 3286, 15006, 20240, 2078, 1012, 22949, 2818, 3489, 1011, 3116, 2012, 1023, 1024, 4700, 7610, 1011, 22851, 2213, 3501, 2869, 7677, 2480, 16932, 16086, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 

In [10]:
tokens = tokenizer.convert_ids_to_tokens(ex_tokenized["input_ids"])
print(tokens)

['[CLS]', 'subject', ':', 'group', 'messaging', 'for', 'admissions', 'process', 'good', 'morning', ',', 'everyone', ',', 'i', 'hope', 'this', 'message', 'finds', 'you', 'well', '.', 'as', 'we', 'continue', 'our', 'admissions', 'processes', ',', 'i', 'would', 'like', 'to', 'update', 'you', 'on', 'the', 'latest', 'developments', 'and', 'key', 'information', '.', 'please', 'find', 'below', 'the', 'timeline', 'for', 'our', 'upcoming', 'meetings', ':', '-', 'w', '##yn', '##q', '##vr', '##h', '##0', '##53', '-', 'meeting', 'at', '10', ':', '20', '##am', '-', 'lu', '##ka', '.', 'bu', '##rg', '-', 'meeting', 'at', '21', '-', 'q', '##ah', '##il', '.', 'wit', '##ta', '##uer', '-', 'meeting', 'at', 'quarter', 'past', '13', '-', 'g', '##hol', '##am', '##hos', '##sei', '##n', '.', 'rus', '##ch', '##ke', '-', 'meeting', 'at', '9', ':', '47', 'pm', '-', 'pd', '##m', '##j', '##rs', '##yo', '##z', '##14', '##60', '[SEP]']


In [31]:
print(ex_tokenized.word_ids())

[None, 0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 52, 52, 52, 52, 52, 52, 53, 54, 55, 56, 57, 58, 58, 59, 60, 60, 61, 62, 62, 63, 64, 65, 66, 67, 68, 68, 68, 69, 70, 70, 70, 71, 72, 73, 74, 75, 76, 77, 78, 78, 78, 78, 78, 78, 79, 80, 80, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 89, 89, 89, 89, 89, 89, 89, None]


In [49]:
zipped = zip(tokens, ex_tokenized.word_ids())
for z in zipped:
    print(z)

('[CLS]', None)
('subject', 0)
(':', 1)
('group', 2)
('messaging', 3)
('for', 4)
('admissions', 5)
('process', 6)
('good', 7)
('morning', 8)
(',', 9)
('everyone', 10)
(',', 11)
('i', 12)
('hope', 13)
('this', 14)
('message', 15)
('finds', 16)
('you', 17)
('well', 18)
('.', 19)
('as', 20)
('we', 21)
('continue', 22)
('our', 23)
('admissions', 24)
('processes', 25)
(',', 26)
('i', 27)
('would', 28)
('like', 29)
('to', 30)
('update', 31)
('you', 32)
('on', 33)
('the', 34)
('latest', 35)
('developments', 36)
('and', 37)
('key', 38)
('information', 39)
('.', 40)
('please', 41)
('find', 42)
('below', 43)
('the', 44)
('timeline', 45)
('for', 46)
('our', 47)
('upcoming', 48)
('meetings', 49)
(':', 50)
('-', 51)
('w', 52)
('##yn', 52)
('##q', 52)
('##vr', 52)
('##h', 52)
('##0', 52)
('##53', 52)
('-', 53)
('meeting', 54)
('at', 55)
('10', 56)
(':', 57)
('20', 58)
('##am', 58)
('-', 59)
('lu', 60)
('##ka', 60)
('.', 61)
('bu', 62)
('##rg', 62)
('-', 63)
('meeting', 64)
('at', 65)
('21', 66)
('-'

In [ ]:
import re
st: str =example["source_text"]
# split text by words and characters
tokens = re.findall(r"\w+|[^\w\s]", st) 
print(tokens[52])

wynqvrh053


In [36]:
pprint(example["privacy_mask"])

[{'end': 297, 'label': 'USERNAME', 'start': 287, 'value': 'wynqvrh053'},
 {'end': 318, 'label': 'TIME', 'start': 311, 'value': '10:20am'},
 {'end': 330, 'label': 'USERNAME', 'start': 321, 'value': 'luka.burg'},
 {'end': 346, 'label': 'TIME', 'start': 344, 'value': '21'},
 {'end': 363, 'label': 'USERNAME', 'start': 349, 'value': 'qahil.wittauer'},
 {'end': 392, 'label': 'TIME', 'start': 377, 'value': 'quarter past 13'},
 {'end': 416,
  'label': 'USERNAME',
  'start': 395,
  'value': 'gholamhossein.ruschke'},
 {'end': 437, 'label': 'TIME', 'start': 430, 'value': '9:47 PM'},
 {'end': 453, 'label': 'USERNAME', 'start': 440, 'value': 'pdmjrsyoz1460'}]


In [41]:
for i in example["privacy_mask"]:
    ex_start = i["start"]
    ex_end = i["end"]
    label = i["label"]

    print(f"label: {label}, example:{example["source_text"][ex_start:ex_end]}")

label: USERNAME, example:wynqvrh053
label: TIME, example:10:20am
label: USERNAME, example:luka.burg
label: TIME, example:21
label: USERNAME, example:qahil.wittauer
label: TIME, example:quarter past 13
label: USERNAME, example:gholamhossein.ruschke
label: TIME, example:9:47 PM
label: USERNAME, example:pdmjrsyoz1460


In [7]:
encoding = tokenizer(
    example["source_text"],
    return_offsets_mapping=True,
    add_special_tokens=False
)

# each token gets a (start, end) char offset
tokens = encoding.tokens()
offsets = encoding["offset_mapping"]  # [(0, 3), (3, 7), ...]
print(tokens)
print(offsets)

['subject', ':', 'group', 'messaging', 'for', 'admissions', 'process', 'good', 'morning', ',', 'everyone', ',', 'i', 'hope', 'this', 'message', 'finds', 'you', 'well', '.', 'as', 'we', 'continue', 'our', 'admissions', 'processes', ',', 'i', 'would', 'like', 'to', 'update', 'you', 'on', 'the', 'latest', 'developments', 'and', 'key', 'information', '.', 'please', 'find', 'below', 'the', 'timeline', 'for', 'our', 'upcoming', 'meetings', ':', '-', 'w', '##yn', '##q', '##vr', '##h', '##0', '##53', '-', 'meeting', 'at', '10', ':', '20', '##am', '-', 'lu', '##ka', '.', 'bu', '##rg', '-', 'meeting', 'at', '21', '-', 'q', '##ah', '##il', '.', 'wit', '##ta', '##uer', '-', 'meeting', 'at', 'quarter', 'past', '13', '-', 'g', '##hol', '##am', '##hos', '##sei', '##n', '.', 'rus', '##ch', '##ke', '-', 'meeting', 'at', '9', ':', '47', 'pm', '-', 'pd', '##m', '##j', '##rs', '##yo', '##z', '##14', '##60']
[(0, 7), (7, 8), (9, 14), (15, 24), (25, 28), (29, 39), (40, 47), (49, 53), (54, 61), (61, 62), (63

In [90]:
pprint(encoding)

{'attention_mask': [1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
                    1,
           

In [8]:
# match padding masks to tokens
from colorama import Fore, Style
for mask in example["privacy_mask"]:
    span_start = mask["start"]
    span_end = mask["end"]
    
    # save every token and offset pair if its part of the privacy mask
    matching_tokens = [
        (tok, off) for tok, off in zip(tokens, offsets)
        if off[0] >= span_start and off[1] <= span_end # save it falls in start:end of mask
    ]
    
    if matching_tokens:
        print(Fore.GREEN + f"mask({span_start}, {span_end}) → {len(matching_tokens)} tokens")
        for tok, off in matching_tokens:
            print(Style.RESET_ALL + f"  {tok!r} at {off}")
    else:
        print(Fore.RED + f"mask({span_start}, {span_end}) — no tokens found")

mask(287, 297) → 7 tokens
  'w' at (287, 288)
  '##yn' at (288, 290)
  '##q' at (290, 291)
  '##vr' at (291, 293)
  '##h' at (293, 294)
  '##0' at (294, 295)
  '##53' at (295, 297)
mask(311, 318) → 4 tokens
  '10' at (311, 313)
  ':' at (313, 314)
  '20' at (314, 316)
  '##am' at (316, 318)
mask(321, 330) → 5 tokens
  'lu' at (321, 323)
  '##ka' at (323, 325)
  '.' at (325, 326)
  'bu' at (326, 328)
  '##rg' at (328, 330)
mask(344, 346) → 1 tokens
  '21' at (344, 346)
mask(349, 363) → 7 tokens
  'q' at (349, 350)
  '##ah' at (350, 352)
  '##il' at (352, 354)
  '.' at (354, 355)
  'wit' at (355, 358)
  '##ta' at (358, 360)
  '##uer' at (360, 363)
mask(377, 392) → 3 tokens
  'quarter' at (377, 384)
  'past' at (385, 389)
  '13' at (390, 392)
mask(395, 416) → 10 tokens
  'g' at (395, 396)
  '##hol' at (396, 399)
  '##am' at (399, 401)
  '##hos' at (401, 404)
  '##sei' at (404, 407)
  '##n' at (407, 408)
  '.' at (408, 409)
  'rus' at (409, 412)
  '##ch' at (412, 414)
  '##ke' at (414, 416

In [74]:
pprint(matching_tokens)

[('pd', (440, 442)),
 ('##m', (442, 443)),
 ('##j', (443, 444)),
 ('##rs', (444, 446)),
 ('##yo', (446, 448)),
 ('##z', (448, 449)),
 ('##14', (449, 451)),
 ('##60', (451, 453))]


In [ ]:
def align_pm_to_tokens(enc_example):
    p_masks = enc_example["privacy_masks"]
    offsets = enc_example["offset_mapping"]
    tokens = tokenizer.convert_ids_to_tokens(enc_example["input_ids"])
    
    for p_mask in p_masks:
        toks_in_p_mask = [(tok, off) for tok, off in zip(tokens, offsets)
                          if tok[0] ]

In [ ]:
def tokenize_and_label(examples):
    encodings = tokenizer(
        examples["source_text"], 
        return_offsets_mappings=True,
        truncation=True,
        add_special_tokens=False
    )
    batch_offsets = encodings["offset_mapping"]
    batch_tokens = tokenizer.convert_ids_to_tokens(encodings["input_ids"])
    batch_masks = examples["privacy_masks"]
    
    for tokens, offsets, masks in zip(batch_tokens, batch_offsets, batch_masks):
        tokens_in_label = [(tok, off) for tok, off in zip(tokens, offsets)
                           if off[0]>= tok[0] and off[1]<= tok[1]] # TODO
        
        for tok in tokens:
            
    

In [ ]:
def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(examples["source_text"], truncation=True)
    id2label = label_info["id2label"]
    
    # list of labels for entire batch
    labels = []
    for word_ids in tokenized_inputs.word_ids():
        previous_word_id = None
        label_ids = []
        for word_id in word_ids:
            # use -100 for special tokens so cross entropy loss ignores them
            if word_id is None:
                label_ids.append(-100)
            elif word_id != previous_word_id:
                label_ids.append(id2label["word_id"])
            else:
                label_ids.append(-100)
            previous_word_id = word_id
        labels.append(label_ids)
    